# 🔎 Actividad 14 — RAG Híbrido con Reranking

## ¿Qué construirás?

Un **asistente que responde preguntas sobre un documento** (la guía del huésped de un hotel ficticio)
usando una técnica profesional llamada **RAG híbrido**: combina dos formas de buscar, fusiona los
resultados, los reordena con un modelo especializado y solo entonces se los pasa al LLM.

```
                         Pregunta del usuario
                                 │
               ┌─────────────────┴─────────────────┐
               ▼                                   ▼
   Búsqueda por SIGNIFICADO             Búsqueda por PALABRAS CLAVE
   (embeddings bge-m3 + coseno)         (BM25)
               │  top 10                           │  top 10
               └─────────────────┬─────────────────┘
                                 ▼
                    Fusión RRF (une las dos listas)
                                 ▼
             Reranker bge-reranker-v2-m3 (reordena con lupa)
                                 ▼  top 3 fragmentos
                  llama3.2 responde SOLO con ese contexto
                                 ▼
                         Respuesta + fuentes
```

## Mapa de la actividad

| Parte | Tema |
|-------|------|
| 1 | Preparar Colab: Ollama, librerías, `bge-m3` y `llama3.2` |
| 2 | Dividir el texto en fragmentos con solape y vectorizarlos |
| 3 | Las dos búsquedas: significado (coseno) y palabras clave (BM25) |
| 4 | Fusión con RRF y reordenamiento con el reranker |
| 5 | Generar la respuesta con `llama3.2` usando solo el contexto |
| 6 | Interfaz Gradio + experimento: 3 tipos de pregunta × 3 tamaños de fragmento |
| 7 | Reflexión |
| 8 | Retos opcionales |
| 9 | Publicación con Ngrok + limpieza + rúbrica |

**⏱️ Duración:** 90–120 minutos · **GPU:** T4 (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*)

> 💡 **Recuerda:** tú solo ejecutas las celdas de arriba a abajo. Lee los comentarios: ahí está la explicación.

## ⚙️ PARTE 1 — Preparar Colab

Instalaremos dos tipos de piezas:

| Pieza | ¿Dónde corre? | ¿Para qué? |
|-------|---------------|------------|
| **Ollama** + `bge-m3` | Servidor local en Colab | Convertir texto en vectores (embeddings) |
| **Ollama** + `llama3.2` | Servidor local en Colab | Redactar la respuesta final |
| **sentence-transformers** + `bge-reranker-v2-m3` | Directamente en Python (GPU) | Reordenar los fragmentos candidatos |
| **Gradio** | Python | La interfaz web |

```
  Tu notebook (Python)
       │  HTTP  ──►  Servidor Ollama (localhost:11434)
       │                ├── bge-m3     → embeddings
       │                └── llama3.2   → texto
       │
       └── en memoria ─► CrossEncoder bge-reranker-v2-m3 (GPU)
```

⏳ Toda la Parte 1 tarda entre 5 y 8 minutos la primera vez (se descargan ~4 GB de modelos).

In [ ]:
# ============================================================
# CELDA 1.1 — Comprobar que tenemos GPU
# ============================================================
# El reranker y los modelos de Ollama van mucho más rápido en GPU.
# nvidia-smi es la herramienta de NVIDIA que muestra la tarjeta gráfica disponible.

!nvidia-smi --query-gpu=name,memory.total --format=csv

# Si ves "Tesla T4" todo está bien.
# Si ves un error: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU, y vuelve a empezar.
print('✅ Revisa arriba que aparezca la GPU (Tesla T4)')

In [ ]:
# ============================================================
# CELDA 1.2 — Instalar Ollama
# ============================================================
# zstd es un descompresor que el instalador de Ollama necesita en Colab.
# -y responde "sí" automáticamente; -q reduce los mensajes.
!apt-get install -y -q zstd > /dev/null

# El script oficial descarga Ollama y lo deja listo como comando de terminal.
# curl -fsSL: descarga en silencio y falla si hay error; "| sh" ejecuta el script descargado.
!curl -fsSL https://ollama.com/install.sh | sh

print('✅ Ollama instalado')

In [ ]:
# ============================================================
# CELDA 1.3 — Instalar las librerías de Python
# ============================================================
# gradio>=5,<6           → interfaz web (fijamos la versión para que el notebook no cambie de comportamiento)
# sentence-transformers  → carga el reranker bge-reranker-v2-m3 como "CrossEncoder"
# pyngrok                → lo usaremos al final para publicar la app
# (numpy, pandas y requests ya vienen instalados en Colab)
!pip install -q 'gradio>=5,<6' sentence-transformers pyngrok

print('✅ Librerías instaladas')

In [ ]:
# ============================================================
# CELDA 1.4 — Arrancar el servidor de Ollama en segundo plano
# ============================================================
import subprocess   # Permite lanzar programas de terminal desde Python
import time         # Para esperar entre intentos
import requests     # Cliente HTTP: así hablaremos con Ollama

# subprocess.Popen lanza el proceso en paralelo (no bloquea el notebook).
# Así el servidor queda corriendo en background mientras continuamos.
# DEVNULL descarta los mensajes del servidor para no saturar la salida.
proceso_ollama = subprocess.Popen(
    ['ollama', 'serve'],           # Equivalente a escribir 'ollama serve' en una terminal
    stdout=subprocess.DEVNULL,     # Descarta la salida estándar
    stderr=subprocess.DEVNULL      # Descarta los mensajes de error
)

OLLAMA_URL = 'http://localhost:11434'   # Puerto por defecto de Ollama

# El servidor tarda unos segundos en arrancar: preguntamos cada segundo (máx. 30 s).
for intento in range(30):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print(f'✅ Servidor Ollama activo (tardó ~{intento} s)')
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)   # Aún no responde: esperamos y reintentamos
else:
    # El bloque else de un for se ejecuta solo si el bucle terminó SIN break.
    print('❌ Ollama no arrancó. Reinicia la sesión y ejecuta desde la celda 1.2.')

In [ ]:
# ============================================================
# CELDA 1.5 — Descargar el modelo de embeddings bge-m3
# ============================================================
# bge-m3 (BAAI) convierte un texto en un vector de 1024 números que resume su significado.
# Es multilingüe (funciona muy bien en español) y acepta textos largos (hasta 8192 tokens).
# Pesa ~1.2 GB.

MODELO_EMBED = 'bge-m3'

# subprocess.run espera a que el comando termine (a diferencia de Popen).
resultado = subprocess.run(['ollama', 'pull', MODELO_EMBED], capture_output=True, text=True)

if resultado.returncode == 0:   # returncode 0 = el comando terminó sin errores
    print(f'✅ {MODELO_EMBED} descargado')
else:
    print(f'❌ Error descargando {MODELO_EMBED}: {resultado.stderr[-300:]}')
    print('   Comprueba que la celda 1.4 mostró "Servidor Ollama activo" y vuelve a ejecutar esta celda.')

In [ ]:
# ============================================================
# CELDA 1.6 — Descargar el modelo generador llama3.2
# ============================================================
# llama3.2 (Meta) en su versión de 3 mil millones de parámetros, cuantizado (~2 GB).
# Es pequeño pero sigue instrucciones bien: ideal para "responde solo con este contexto".

MODELO_LLM = 'llama3.2'

resultado = subprocess.run(['ollama', 'pull', MODELO_LLM], capture_output=True, text=True)

if resultado.returncode == 0:
    print(f'✅ {MODELO_LLM} descargado')
else:
    print(f'❌ Error descargando {MODELO_LLM}: {resultado.stderr[-300:]}')
    print('   Vuelve a ejecutar esta celda; las descargas a veces se cortan.')

In [ ]:
# ============================================================
# CELDA 1.7 — Verificar los modelos disponibles en Ollama
# ============================================================
# /api/tags devuelve la lista de modelos descargados en el servidor.

modelos = [m['name'] for m in requests.get(f'{OLLAMA_URL}/api/tags').json()['models']]
print('Modelos en Ollama:', modelos)

# Los nombres llegan con etiqueta (ej. 'bge-m3:latest'), por eso usamos startswith.
faltan = [m for m in (MODELO_EMBED, MODELO_LLM) if not any(x.startswith(m) for x in modelos)]

if faltan:
    print(f'❌ Faltan: {faltan}. Repite la celda de descarga correspondiente.')
else:
    print('✅ PARTE 1 completa: bge-m3 y llama3.2 listos')

## ✂️ PARTE 2 — Dividir el texto en fragmentos y vectorizarlos

Un LLM no puede leer un documento entero en cada pregunta (sería lento y caro), así que lo
**partimos en fragmentos** (*chunks*) y luego solo le pasamos los más útiles.

### ¿Por qué con solape?

Si cortamos "a ciegas", una idea puede quedar partida entre dos fragmentos.
El **solape** repite las últimas palabras de un fragmento al inicio del siguiente:

```
Texto:   [w1 w2 w3 w4 w5 w6 w7 w8 w9 w10 w11 w12 ...]

tam=6, solape=2   →   paso = tam - solape = 4

F0:  w1  w2  w3  w4  w5  w6
F1:              w5  w6  w7  w8  w9  w10      ← w5 y w6 se repiten
F2:                          w9  w10 w11 w12 ...
```

### El dilema del tamaño

| Fragmentos **pequeños** | Fragmentos **grandes** |
|---|---|
| ✅ Muy precisos: hablan de una sola cosa | ✅ Mucho contexto alrededor |
| ❌ Pueden perder contexto necesario | ❌ Mezclan temas y "diluyen" el significado |

En la Parte 6 lo comprobarás experimentalmente.

### Vectorizar

Después, cada fragmento se convierte en un **embedding** con `bge-m3`: una lista de 1024 números.
Textos con significado parecido producen vectores que "apuntan" en direcciones parecidas.

In [ ]:
# ============================================================
# CELDA 2.1 — Cargar el documento de trabajo
# ============================================================
# Usamos la guía del huésped de un hotel ficticio. Está escrita dentro del notebook para que
# la actividad no dependa de descargas externas. Tiene datos muy concretos (horarios, precios,
# contraseñas) que nos permitirán comprobar si el sistema encuentra la información correcta.

DOCUMENTO = """
GUÍA DEL HUÉSPED — HOTEL MIRADOR DEL LAGO

BIENVENIDA. El Hotel Mirador del Lago es un hotel de cuatro estrellas situado en la orilla norte del lago, a veinte minutos en coche del centro de la ciudad. Dispone de 86 habitaciones, de las cuales 4 están adaptadas para personas con movilidad reducida, con plato de ducha a ras de suelo, barras de apoyo y puertas de 90 centímetros. Esta guía reúne las normas y servicios del hotel. Si tiene cualquier duda, la recepción está abierta las 24 horas y puede llamar marcando el 9 desde el teléfono de su habitación.

LLEGADA Y SALIDA. La entrada a las habitaciones (check-in) es a partir de las 15:00 y la salida (check-out) debe hacerse antes de las 12:00. Si necesita quedarse más tiempo, puede solicitar la salida tardía hasta las 14:00 con un suplemento de 25 euros, sujeto a disponibilidad. Si llega antes de las 15:00, puede dejar su equipaje en la consigna gratuita junto a recepción y disfrutar de las zonas comunes. Para registrarse es obligatorio presentar un documento de identidad o pasaporte de cada adulto.

DESAYUNO. El desayuno buffet se sirve en el restaurante La Orilla, en la planta baja, todos los días de 7:00 a 10:30. Incluye productos locales, pan recién horneado, fruta de temporada y una estación de huevos al momento. Hay opciones sin gluten, vegetarianas y veganas debidamente señalizadas. Si su tarifa no incluye desayuno, puede añadirlo en recepción por 18 euros por persona; los niños menores de 6 años desayunan gratis.

CONEXIÓN WIFI. Todo el hotel tiene cobertura WiFi gratuita. La red se llama MiradorGuest y la contraseña es Lago2026! (con signo de exclamación final). Para videollamadas o descargas grandes recomendamos la red MiradorPremium, disponible en recepción por 5 euros al día. En la sala de reuniones Bruma hay conexión por cable.

MASCOTAS. Aceptamos perros y gatos de hasta 10 kilos de peso, con un máximo de un animal por habitación y un suplemento de 15 euros por noche. Los animales no pueden acceder al restaurante, a la piscina ni al spa. Al hacer la reserva debe indicar que viaja con su mascota, ya que solo algunas habitaciones de la planta primera están habilitadas para ello. Ofrecemos cama, comedero y bebedero sin coste adicional.

PISCINA Y SPA. La piscina climatizada interior abre de 8:00 a 21:00 y el uso del gorro de baño es obligatorio. Los menores de 14 años deben ir acompañados de un adulto. El spa, con sauna, baño turco y circuito de chorros, abre de 10:00 a 20:00 y requiere reserva previa llamando a la extensión 214. Los masajes se reservan con al menos 24 horas de antelación. El acceso al spa es solo para mayores de 16 años.

RESTAURANTE. El restaurante La Orilla ofrece cocina de mercado con vistas al lago. El almuerzo se sirve de 13:00 a 16:00 y la cena de 20:00 a 23:00. Los jueves por la noche hay menú degustación de pescado del lago por 45 euros. Recomendamos reservar mesa para la cena en recepción, especialmente los fines de semana. Si tiene alergias o intolerancias alimentarias, comuníquelo al personal de sala antes de pedir.

APARCAMIENTO Y TRANSPORTE. El hotel dispone de aparcamiento subterráneo vigilado por 12 euros al día, con 4 plazas equipadas con cargador para vehículos eléctricos que se asignan por orden de llegada. Ofrecemos traslado al aeropuerto en furgoneta por 35 euros por trayecto, que debe solicitarse con al menos 24 horas de antelación. Un autobús urbano, la línea 7, para frente a la entrada del hotel cada 30 minutos y llega al centro de la ciudad en 25 minutos.

ACTIVIDADES. Los huéspedes pueden usar gratuitamente las bicicletas del hotel, disponibles en el cobertizo junto al embarcadero de 9:00 a 19:00; los cascos se piden en recepción. Los martes y jueves a las 10:00 sale una excursión en barco por el lago de dos horas de duración, con un precio de 20 euros por persona. Las plazas son limitadas a 12 personas, así que conviene apuntarse el día anterior. Junto al hotel comienza un sendero señalizado de 6 kilómetros que rodea la bahía.

BAR Y TERRAZA. El bar Mirador, en la terraza de la primera planta, abre de 11:00 a 24:00. Sirve cócteles, vinos de la región, zumos naturales y una carta de tapas y bocadillos durante todo su horario. Los viernes y sábados hay música en directo de 21:00 a 23:00. En caso de lluvia, el servicio se traslada al salón interior.

SERVICIO DE HABITACIONES. El servicio de habitaciones funciona las 24 horas con una carta reducida de platos fríos, sándwiches y bebidas; de 13:00 a 23:00 también puede pedir platos calientes de la carta del restaurante. Se aplica un cargo de servicio de 4 euros por pedido. La limpieza de la habitación se realiza diariamente entre las 9:00 y las 14:00; si no desea ser molestado, cuelgue el cartel en la puerta. Las toallas solo se cambian si las deja en el suelo de la bañera, como parte de nuestro compromiso con el medio ambiente.

NORMAS DE CONVIVENCIA. Entre las 23:00 y las 8:00 rige el horario de descanso y se ruega mantener el silencio en pasillos y habitaciones. Está prohibido fumar en todo el edificio, incluidos los balcones; hacerlo conlleva una penalización de 150 euros por limpieza especial. No se permiten visitas en las habitaciones después de las 22:00. Las fiestas o celebraciones deben organizarse en los salones del hotel, previa consulta con el departamento de eventos.

OBJETOS OLVIDADOS. Todo lo que los huéspedes dejen en la habitación tras su salida se custodia en recepción durante 90 días. Para recuperarlo, escriba a perdidos@miradordellago.example indicando su nombre, el número de habitación y las fechas de la estancia. Se lo enviamos por mensajería a portes debidos o puede pasar a recogerlo en persona. Pasado ese plazo, los artículos se donan a una organización benéfica local.

CANCELACIONES Y PAGOS. Las reservas con tarifa flexible pueden cancelarse sin coste hasta 48 horas antes de la fecha de llegada; después se cobra la primera noche. Las tarifas no reembolsables no admiten cambios ni cancelaciones. Aceptamos tarjetas de crédito y débito, transferencia bancaria y pago en efectivo. A la llegada se solicita una tarjeta como garantía para los gastos extra, que se liquidan en el momento de la salida.
""".strip()   # .strip() quita los saltos de línea sobrantes del principio y del final

# len(texto.split()) cuenta palabras: split() separa por espacios y saltos de línea.
print(f'✅ Documento cargado: {len(DOCUMENTO.split())} palabras, {len(DOCUMENTO)} caracteres')
print(DOCUMENTO[:300] + ' ...')

In [ ]:
# ============================================================
# CELDA 2.2 — Definir la función que divide el texto en fragmentos con solape
# ============================================================
# Medimos el tamaño en PALABRAS (no en caracteres) para no cortar palabras por la mitad
# y porque es más intuitivo. Los sistemas en producción suelen medirlo en tokens.

def dividir_en_fragmentos(texto, tam=120, solape=20):
    """Devuelve una lista de fragmentos de `tam` palabras que se solapan `solape` palabras."""
    if solape >= tam:
        # Si el solape fuera igual o mayor que el tamaño, el paso sería 0 o negativo
        # y el bucle nunca avanzaría.
        raise ValueError(f'El solape ({solape}) debe ser menor que el tamaño ({tam}).')

    palabras = texto.split()   # Lista de palabras del documento
    paso = tam - solape        # Cuántas palabras avanzamos entre un fragmento y el siguiente
    fragmentos = []

    # range(inicio, fin, paso) genera 0, paso, 2*paso, ... → posición inicial de cada fragmento
    for inicio in range(0, len(palabras), paso):
        trozo = palabras[inicio:inicio + tam]   # Rebanada de 'tam' palabras (o menos al final)
        fragmentos.append(' '.join(trozo))      # Volvemos a unir las palabras con espacios
        if inicio + tam >= len(palabras):       # Este trozo ya llegó al final del texto:
            break                               # paramos para no crear fragmentos repetidos

    return fragmentos

print('✅ Función dividir_en_fragmentos definida')

In [ ]:
# ============================================================
# CELDA 2.3 — Probar la división con un ejemplo pequeño
# ============================================================
# Antes de usarla con el documento real, la probamos con 12 "palabras" numeradas
# para VER el solape. Debe coincidir con el diagrama del bloque anterior.

ejemplo = ' '.join(f'w{i}' for i in range(1, 13))   # 'w1 w2 w3 ... w12'

for i, f in enumerate(dividir_en_fragmentos(ejemplo, tam=6, solape=2)):
    print(f'F{i}: {f}')

print('✅ Observa cómo cada fragmento repite las 2 últimas palabras del anterior')

In [ ]:
# ============================================================
# CELDA 2.4 — Dividir el documento real
# ============================================================
# Valores iniciales: fragmentos de 120 palabras con 20 de solape (≈17 %).
# Es un punto de partida razonable; en la Parte 6 los cambiaremos.

TAM_FRAGMENTO = 120
SOLAPE = 20

fragmentos = dividir_en_fragmentos(DOCUMENTO, TAM_FRAGMENTO, SOLAPE)

print(f'✅ {len(fragmentos)} fragmentos creados\n')
# Mostramos el inicio de los 3 primeros para hacernos una idea
for i, f in enumerate(fragmentos[:3]):
    print(f'── F{i} ({len(f.split())} palabras): {f[:160]}...\n')

In [ ]:
# ============================================================
# CELDA 2.5 — Definir la función que vectoriza textos con bge-m3
# ============================================================
import numpy as np   # Librería de cálculo numérico: guardaremos los vectores como matriz

def vectorizar(textos, lote=16):
    """Convierte una lista de textos en una matriz (n_textos × 1024) de embeddings."""
    vectores = []
    # Enviamos los textos en lotes de 16: una sola petición con cientos de textos
    # podría tardar demasiado y superar el timeout.
    for i in range(0, len(textos), lote):
        r = requests.post(
            f'{OLLAMA_URL}/api/embed',                         # Endpoint de embeddings de Ollama
            json={'model': MODELO_EMBED,                       # bge-m3
                  'input': textos[i:i + lote]},                # Acepta una lista de textos a la vez
            timeout=300                                        # Hasta 5 min por si el modelo se está cargando
        )
        r.raise_for_status()                    # Lanza un error claro si Ollama respondió con fallo
        vectores.extend(r.json()['embeddings'])  # Lista de vectores, uno por texto
    # float32 ocupa la mitad que float64 y sobra precisión para comparar significados
    return np.array(vectores, dtype=np.float32)

print('✅ Función vectorizar definida')

In [ ]:
# ============================================================
# CELDA 2.6 — Probar la vectorización con tres frases
# ============================================================
# Comprobamos que funciona y vemos cómo luce un embedding.
# La primera llamada tarda unos segundos porque Ollama carga bge-m3 en la GPU.

prueba = vectorizar(['El perro duerme en el sofá',
                     'Un can descansa en el sillón',
                     'La factura se paga con tarjeta'])

print(f'Forma de la matriz: {prueba.shape}  → (3 textos, {prueba.shape[1]} dimensiones)')
print(f'Primeros 6 números del primer vector: {np.round(prueba[0][:6], 3)}')
print('✅ bge-m3 responde correctamente')

In [ ]:
# ============================================================
# CELDA 2.7 — Vectorizar todos los fragmentos del documento
# ============================================================
# Este es el "índice vectorial": una fila por fragmento.
# En producción se guardaría en una base de datos vectorial (Chroma, Qdrant, pgvector...);
# aquí basta con una matriz de numpy porque el documento es pequeño.

inicio = time.time()
vectores = vectorizar(fragmentos)

print(f'✅ Índice vectorial listo: {vectores.shape[0]} fragmentos × {vectores.shape[1]} dimensiones '
      f'({time.time() - inicio:.1f} s)')

## 🔍 PARTE 3 — Las dos búsquedas

Cada búsqueda tiene un "superpoder" distinto, y por eso las combinaremos:

| | Búsqueda **semántica** (coseno) | Búsqueda **léxica** (BM25) |
|---|---|---|
| Compara | Significados (vectores) | Palabras exactas |
| Gana cuando... | La pregunta usa **otras palabras**: "cuarto" vs "habitación" | Hay **términos raros o exactos**: "WiFi", "MiradorGuest", "214" |
| Falla cuando... | Hay códigos, nombres propios o números | Hay sinónimos o paráfrasis |

### Similitud coseno

Mide el ángulo entre dos vectores (1 = misma dirección, 0 = nada que ver):

```
                 A · B              (producto punto)
  cos(θ)  =  ───────────
              |A| · |B|             (producto de las longitudes)
```

### BM25 (Best Matching 25)

Es el algoritmo clásico de los buscadores (Elasticsearch, Lucene). Un fragmento puntúa alto si:

1. Contiene las palabras de la pregunta **varias veces** (frecuencia, con saturación: la 10ª repetición suma poco)
2. Esas palabras son **raras** en el documento (IDF: "contraseña" vale más que "hotel")
3. El fragmento **no es excesivamente largo** (normalización por longitud)

In [ ]:
# ============================================================
# CELDA 3.1 — Definir la búsqueda semántica por similitud coseno
# ============================================================

def busqueda_semantica(pregunta, vectores, n=10):
    """Devuelve [(índice_fragmento, similitud), ...] con los n fragmentos más parecidos."""
    q = vectorizar([pregunta])[0]   # Vector de la pregunta (con el MISMO modelo que los fragmentos)

    # vectores @ q multiplica la matriz por el vector: calcula el producto punto
    # de la pregunta con TODOS los fragmentos de una vez (mucho más rápido que un bucle).
    # Dividimos por las normas (longitudes) para obtener el coseno.
    # El 1e-10 evita dividir por cero si algún vector fuera todo ceros.
    similitudes = (vectores @ q) / (np.linalg.norm(vectores, axis=1) * np.linalg.norm(q) + 1e-10)

    # argsort ordena de menor a mayor; con el signo menos obtenemos de mayor a menor.
    orden = np.argsort(-similitudes)[:n]
    return [(int(i), float(similitudes[i])) for i in orden]

print('✅ Función busqueda_semantica definida')

In [ ]:
# ============================================================
# CELDA 3.2 — Probar la búsqueda semántica con una paráfrasis
# ============================================================
# La pregunta NO usa las palabras del documento ("cuarto", "recupero", "móvil"),
# pero el significado es el de la sección OBJETOS OLVIDADOS.

pregunta_prueba = 'Me dejé el cargador del móvil en el cuarto al irme, ¿cómo lo recupero?'

for idx, sim in busqueda_semantica(pregunta_prueba, vectores, n=3):
    print(f'F{idx:<3} coseno={sim:.3f}  │ {fragmentos[idx][:110]}...')

print('\n✅ ¿Aparece la sección de OBJETOS OLVIDADOS arriba? Eso es entender el significado.')

In [ ]:
# ============================================================
# CELDA 3.3 — Definir el tokenizador para BM25
# ============================================================
# BM25 trabaja con palabras, así que necesitamos "limpiar" el texto igual
# en los fragmentos y en la pregunta. Si no, "WiFi" y "wifi" serían palabras distintas.
import re            # Expresiones regulares: para extraer palabras
import unicodedata   # Para quitar tildes

# Palabras vacías (stopwords): muy frecuentes y sin significado propio.
# Si no las quitamos, "de", "la" o "el" harían coincidir casi cualquier fragmento.
STOPWORDS = set("""
a al algo como con cual cuales de del desde donde el en entre era es esa ese esta este
esto estos ha hay la las le les lo los mas me mi mis muy ni no nos o os para pero por puede
que se si sin sobre su sus te tiene tu un una uno unos y ya yo
""".split())

def tokenizar(texto):
    """Minúsculas → sin tildes → solo palabras → sin stopwords."""
    texto = texto.lower()
    # NFD separa cada letra de su tilde ('á' → 'a' + '´'); luego descartamos las tildes
    # (categoría 'Mn' = marcas diacríticas). Así "habitación" y "habitacion" coinciden.
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                    if unicodedata.category(c) != 'Mn')
    # \w+ captura secuencias de letras/números: descarta signos de puntuación
    palabras = re.findall(r'\w+', texto)
    return [p for p in palabras if p not in STOPWORDS]

print(tokenizar('¿Cuál es la contraseña del WiFi de la habitación?'))
print('✅ Tokenizador listo')

In [ ]:
# ============================================================
# CELDA 3.4 — Implementar BM25 desde cero
# ============================================================
# Existen librerías (rank_bm25, Elasticsearch), pero programarlo nos deja ver la fórmula.
import math
from collections import Counter   # Cuenta cuántas veces aparece cada elemento de una lista

class BM25:
    def __init__(self, fragmentos, k1=1.5, b=0.75):
        # k1 controla la SATURACIÓN: cuánto premia repetir una palabra (1.2–2.0 es lo típico).
        # b controla la normalización por LONGITUD: 0 = ignorarla, 1 = penalizar textos largos al máximo.
        self.k1, self.b = k1, b
        self.docs = [tokenizar(f) for f in fragmentos]               # Fragmentos tokenizados
        self.N = len(self.docs)                                      # Número de fragmentos
        self.largos = [len(d) for d in self.docs]                    # Palabras útiles de cada uno
        self.largo_medio = sum(self.largos) / max(self.N, 1)         # Largo promedio
        self.tf = [Counter(d) for d in self.docs]                    # Frecuencia de cada palabra por fragmento
        # df = en cuántos fragmentos aparece cada palabra (set() evita contar repeticiones)
        self.df = Counter(p for d in self.docs for p in set(d))

    def idf(self, palabra):
        # Palabra rara (df pequeño) → idf alto. El +1 dentro del log evita valores negativos.
        df = self.df.get(palabra, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def puntuar(self, pregunta):
        """Devuelve una puntuación BM25 por cada fragmento."""
        q = tokenizar(pregunta)
        puntuaciones = []
        for tf, largo in zip(self.tf, self.largos):
            total = 0.0
            for palabra in q:
                f = tf.get(palabra, 0)              # Veces que la palabra aparece en este fragmento
                if f == 0:
                    continue                        # No aparece: no suma nada
                norm = 1 - self.b + self.b * largo / self.largo_medio   # Penalización por longitud
                total += self.idf(palabra) * f * (self.k1 + 1) / (f + self.k1 * norm)
            puntuaciones.append(total)
        return puntuaciones

print('✅ Clase BM25 definida')

In [ ]:
# ============================================================
# CELDA 3.5 — Definir la búsqueda por palabras clave
# ============================================================

def busqueda_bm25(pregunta, bm25, n=10):
    """Devuelve [(índice_fragmento, puntuación), ...] con los n fragmentos mejor puntuados."""
    puntos = bm25.puntuar(pregunta)
    orden = sorted(range(len(puntos)), key=lambda i: -puntos[i])[:n]   # De mayor a menor
    # Descartamos los de puntuación 0: no comparten ni una palabra con la pregunta
    return [(i, puntos[i]) for i in orden if puntos[i] > 0]

bm25 = BM25(fragmentos)   # Construimos el índice léxico sobre los mismos fragmentos
print(f'✅ Índice BM25 listo: {len(bm25.df)} palabras distintas en {bm25.N} fragmentos')

In [ ]:
# ============================================================
# CELDA 3.6 — Comparar las dos búsquedas con dos preguntas
# ============================================================
# Pregunta 1: tiene palabras exactas y poco frecuentes → terreno de BM25.
# Pregunta 2: es una paráfrasis → terreno de la búsqueda semántica.

def mostrar(titulo, resultados):
    print(f'  {titulo}')
    for idx, s in resultados[:3]:
        print(f'    F{idx:<3} {s:6.3f} │ {fragmentos[idx][:85]}...')

for p in ['¿Cuál es la contraseña del WiFi?', pregunta_prueba]:
    print(f'\n❓ {p}')
    mostrar('Semántica (coseno):', busqueda_semantica(p, vectores))
    resultado_bm25 = busqueda_bm25(p, bm25)
    mostrar('Léxica (BM25):', resultado_bm25) if resultado_bm25 else print('  Léxica (BM25): ⚠️ ninguna coincidencia')

print('\n✅ Compara: ¿en qué pregunta acierta cada método? Por eso los combinaremos.')

In [ ]:
# ============================================================
# CELDA 3.7 — Empaquetar todo en un "índice" reutilizable
# ============================================================
# En la Parte 6 cambiaremos el tamaño de fragmento muchas veces. Para no repetir pasos,
# esta función crea fragmentos + vectores + BM25 de una sola vez.
# El diccionario _cache guarda los índices ya calculados: si pedimos la misma
# combinación (texto, tam, solape) otra vez, no se vuelve a vectorizar.

_cache = {}

def construir_indice(texto, tam, solape):
    clave = (hash(texto), tam, solape)   # hash() resume el texto en un número
    if clave not in _cache:
        frags = dividir_en_fragmentos(texto, tam, solape)
        _cache[clave] = {'fragmentos': frags,
                         'vectores': vectorizar(frags),
                         'bm25': BM25(frags)}
    return _cache[clave]

indice = construir_indice(DOCUMENTO, TAM_FRAGMENTO, SOLAPE)
print(f'✅ PARTE 3 completa: índice híbrido con {len(indice["fragmentos"])} fragmentos')

## 🔀 PARTE 4 — Fusión con RRF y reordenamiento con el reranker

### Problema: las puntuaciones no se pueden sumar

El coseno va de -1 a 1; BM25 puede valer 0, 3 o 17. Sumarlas sería mezclar peras con manzanas.
**Reciprocal Rank Fusion (RRF)** ignora las puntuaciones y usa solo la **posición** en cada lista:

```
                    1
  RRF(frag) =  Σ  ─────────          k = 60 (valor estándar que suaviza las diferencias)
              listas  k + posición

  Ejemplo:          Semántica   BM25     RRF
  F7                  1º         3º      1/61 + 1/63 = 0.0323   ← está bien en las dos: gana
  F2                  2º         —       1/62        = 0.0161
  F9                  —          1º      1/61        = 0.0164
```

### Reranker: la "segunda lectura"

Los embeddings comparan pregunta y fragmento **por separado** (cada uno tiene su vector).
Un **reranker** (*cross-encoder*) lee **pregunta + fragmento juntos** y decide qué tan bien
lo responde. Es mucho más preciso, pero también más lento, por eso solo lo aplicamos a los
~10 candidatos que dejó RRF, no a todo el documento.

```
  Bi-encoder (bge-m3)                 Cross-encoder (bge-reranker-v2-m3)
  pregunta ──► [vector] ┐             [pregunta + fragmento] ──► modelo ──► 0.97
  fragmento ─► [vector] ┴► coseno
  rápido, aproximado                  lento, preciso
```

In [ ]:
# ============================================================
# CELDA 4.1 — Definir la fusión RRF
# ============================================================
from collections import defaultdict   # Diccionario que crea el valor 0.0 si la clave no existe

def fusion_rrf(listas, k=60):
    """Recibe varias listas [(idx, puntuación), ...] y devuelve una sola, ordenada por RRF."""
    puntos = defaultdict(float)
    for lista in listas:
        # enumerate(..., start=1) da la posición 1, 2, 3... en lugar de 0, 1, 2...
        for posicion, (idx, _) in enumerate(lista, start=1):
            puntos[idx] += 1 / (k + posicion)   # La puntuación original (_) se ignora a propósito
    return sorted(puntos.items(), key=lambda x: -x[1])   # De mayor a menor

print('✅ Función fusion_rrf definida')

In [ ]:
# ============================================================
# CELDA 4.2 — Probar RRF con listas inventadas
# ============================================================
# Reproducimos el ejemplo de la tabla anterior para comprobar que la fórmula está bien.

semantica_falsa = [(7, 0.91), (2, 0.85), (4, 0.70)]   # F7 es el 1º en significado
bm25_falso      = [(9, 12.0), (5, 8.0), (7, 6.0)]      # F7 es el 3º en palabras clave

for idx, p in fusion_rrf([semantica_falsa, bm25_falso]):
    print(f'F{idx}: {p:.4f}')

print('✅ F7 queda primero porque aparece en AMBAS listas')

In [ ]:
# ============================================================
# CELDA 4.3 — Cargar el reranker bge-reranker-v2-m3
# ============================================================
# Este modelo no está en Ollama, así que lo cargamos con sentence-transformers desde
# HuggingFace (~2.2 GB, tarda 1-2 min la primera vez).
import torch
from sentence_transformers import CrossEncoder

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'   # GPU si existe

reranker = CrossEncoder(
    'BAAI/bge-reranker-v2-m3',   # Reranker multilingüe de la misma familia que bge-m3
    max_length=512,              # Máx. tokens de pregunta+fragmento (lo demás se recorta)
    device=dispositivo
)

if dispositivo == 'cuda':
    # float16 = media precisión: usa la mitad de memoria y va casi el doble de rápido en la T4,
    # con una pérdida de exactitud imperceptible para ordenar fragmentos.
    reranker.model.half()

print(f'✅ Reranker cargado en {dispositivo.upper()}')

In [ ]:
# ============================================================
# CELDA 4.4 — Probar el reranker por separado
# ============================================================
# predict() recibe pares (pregunta, texto) y devuelve una puntuación por par:
# cuanto más alta, mejor responde ese texto a la pregunta.

pares = [
    ('¿A qué hora es el desayuno?', 'El desayuno buffet se sirve de 7:00 a 10:30.'),
    ('¿A qué hora es el desayuno?', 'La cena se sirve de 20:00 a 23:00.'),
    ('¿A qué hora es el desayuno?', 'El aparcamiento cuesta 12 euros al día.'),
]
for (preg, texto), s in zip(pares, reranker.predict(pares)):
    print(f'{s:7.3f} │ {texto}')

print('✅ El reranker distingue "desayuno" de "cena" aunque ambas frases hablen de horarios de comida')

In [ ]:
# ============================================================
# CELDA 4.5 — Definir el pipeline completo de recuperación híbrida
# ============================================================
# Une todo lo anterior: 2 búsquedas → RRF → reranker → top_k.
# Devuelve también los pasos intermedios para poder mostrarlos en la interfaz.

def recuperar(pregunta, indice, n_candidatos=10, top_k=3):
    frags = indice['fragmentos']

    # 1) Las dos búsquedas, cada una con sus n mejores candidatos
    sem = busqueda_semantica(pregunta, indice['vectores'], n=n_candidatos)
    lex = busqueda_bm25(pregunta, indice['bm25'], n=n_candidatos)

    # 2) Fusión: una sola lista; nos quedamos con los n_candidatos primeros
    fusion = fusion_rrf([sem, lex])[:n_candidatos]

    # 3) Reranking: el cross-encoder puntúa cada candidato junto a la pregunta
    pares = [(pregunta, frags[idx]) for idx, _ in fusion]
    puntos = reranker.predict(pares)
    reordenados = sorted(zip([idx for idx, _ in fusion], puntos), key=lambda x: -x[1])

    return {'semantica': sem, 'bm25': lex, 'rrf': fusion,
            'final': [(int(i), float(s)) for i, s in reordenados[:top_k]]}

print('✅ Función recuperar definida')

In [ ]:
# ============================================================
# CELDA 4.6 — Probar el pipeline y ver cómo cambia el orden en cada etapa
# ============================================================

r = recuperar('¿Puedo llevar a mi perro a la piscina?', indice)

# Para cada etapa mostramos solo los índices de fragmento, en orden
for etapa in ('semantica', 'bm25', 'rrf', 'final'):
    print(f'{etapa:>10}: ' + ', '.join(f'F{i}' for i, _ in r[etapa][:5]))

idx_top = r['final'][0][0]
print(f'\n🥇 Fragmento ganador F{idx_top}: {indice["fragmentos"][idx_top][:200]}...')
print('✅ PARTE 4 completa: recuperación híbrida funcionando')

## 🦙 PARTE 5 — Generar la respuesta con llama3.2 usando SOLO el contexto

Ya tenemos los 3 mejores fragmentos. Ahora el LLM **redacta** la respuesta, pero con reglas estrictas:

1. Usar únicamente la información del contexto (nada de su "conocimiento general")
2. Si la respuesta no está, decirlo con una frase fija en vez de inventar (**alucinar**)
3. Citar qué fragmentos usó: `[F1]`, `[F2]`...

```
  ┌─ system ─────────────────────────────────────────┐
  │ Reglas: solo contexto, frase fija si no sabes... │
  └──────────────────────────────────────────────────┘
  ┌─ user ───────────────────────────────────────────┐
  │ CONTEXTO:                                        │
  │ [F1] ...texto del fragmento 1...                 │
  │ [F2] ...                                         │
  │ PREGUNTA: ¿...?                                  │
  └──────────────────────────────────────────────────┘
                         ▼
                     llama3.2
```

In [ ]:
# ============================================================
# CELDA 5.1 — Definir las instrucciones del sistema y el constructor del prompt
# ============================================================

NO_ENCONTRADO = 'No encuentro esa información en el documento.'

# El mensaje "system" fija el comportamiento del modelo para toda la conversación.
SISTEMA = f"""Eres el asistente del Hotel Mirador del Lago.
Responde a la pregunta usando ÚNICAMENTE la información del CONTEXTO.
- Si la respuesta no está en el contexto, responde exactamente: "{NO_ENCONTRADO}"
- No inventes datos, precios ni horarios.
- Responde en español, de forma breve y clara.
- Al final indica entre corchetes los fragmentos que usaste, por ejemplo [F1][F3]."""

def construir_prompt(pregunta, textos):
    """Numera los fragmentos como [F1], [F2]... para que el modelo pueda citarlos."""
    contexto = '\n\n'.join(f'[F{i}] {t}' for i, t in enumerate(textos, start=1))
    return f'CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}'

# Vemos un prompt de ejemplo con fragmentos cortos inventados
print(construir_prompt('¿Hay piscina?', ['La piscina abre de 8:00 a 21:00.', 'El spa abre a las 10:00.']))
print('\n✅ Prompt listo')

In [ ]:
# ============================================================
# CELDA 5.2 — Definir la llamada a llama3.2
# ============================================================

def generar_respuesta(pregunta, textos):
    r = requests.post(
        f'{OLLAMA_URL}/api/chat',                 # Endpoint de chat (mensajes con roles)
        json={
            'model': MODELO_LLM,
            'messages': [
                {'role': 'system', 'content': SISTEMA},
                {'role': 'user',   'content': construir_prompt(pregunta, textos)},
            ],
            'stream': False,                      # Esperamos la respuesta completa de una vez
            'options': {
                'temperature': 0.1,   # Casi determinista: en RAG queremos fidelidad, no creatividad
                'num_ctx': 4096,      # Ventana de contexto en tokens: cabe de sobra el prompt con 6 fragmentos
            },
        },
        timeout=300,
    )
    r.raise_for_status()
    return r.json()['message']['content'].strip()

print('✅ Función generar_respuesta definida')

In [ ]:
# ============================================================
# CELDA 5.3 — Probar el RAG completo con una pregunta que SÍ está y otra que NO
# ============================================================
# La segunda pregunta (niñera) no aparece en la guía: el modelo debería negarse a inventar.

def rag(pregunta, indice, top_k=3):
    """Recupera los mejores fragmentos y genera la respuesta. Devuelve (respuesta, recuperación)."""
    rec = recuperar(pregunta, indice, top_k=top_k)
    textos = [indice['fragmentos'][i] for i, _ in rec['final']]
    return generar_respuesta(pregunta, textos), rec

for p in ['¿A qué hora tengo que dejar la habitación y puedo salir más tarde?',
          '¿Cuánto cuesta el servicio de niñera?']:
    inicio = time.time()
    respuesta, _ = rag(p, indice)
    print(f'❓ {p}\n💬 {respuesta}\n   ({time.time() - inicio:.1f} s)\n')

print('✅ PARTE 5 completa. ¿La segunda respuesta reconoce que no tiene la información?')

## 🧪 PARTE 6 — Experimento y interfaz Gradio

Probaremos **tres tipos de pregunta**, cada uno pone a prueba algo distinto:

| Tipo | Pregunta | ¿Qué pone a prueba? |
|------|----------|---------------------|
| 🔤 **Literal** | ¿Cuál es la contraseña del WiFi? | Coincidencia exacta de palabras → BM25 |
| 💭 **Paráfrasis** | Me dejé el cargador del móvil en el cuarto al irme... | Entender el significado → embeddings |
| 🧩 **Síntesis** | ¿Qué opciones hay para comer en el hotel y en qué horarios? | Juntar información repartida en varias secciones |

Y las repetiremos con **tres tamaños de fragmento** (40, 120 y 300 palabras) para ver cómo
cambia la calidad. Después montaremos la interfaz para que sigas experimentando tú.

In [ ]:
# ============================================================
# CELDA 6.1 — Definir las tres preguntas de prueba
# ============================================================

PREGUNTAS_PRUEBA = {
    '🔤 Literal':    '¿Cuál es la contraseña del WiFi?',
    '💭 Paráfrasis': 'Me dejé el cargador del móvil en el cuarto al irme, ¿cómo lo recupero?',
    '🧩 Síntesis':   '¿Qué opciones hay para comer en el hotel y en qué horarios?',
}
TAMANOS = [40, 120, 300]   # Pequeño, mediano y grande (en palabras)

print(f'✅ {len(PREGUNTAS_PRUEBA)} preguntas × {len(TAMANOS)} tamaños = '
      f'{len(PREGUNTAS_PRUEBA) * len(TAMANOS)} combinaciones')

In [ ]:
# ============================================================
# CELDA 6.2 — Ejecutar el experimento (≈1-2 minutos)
# ============================================================
import pandas as pd   # Para mostrar los resultados como tabla

filas = []
for tam in TAMANOS:
    # Solape del 15 % del tamaño: así es proporcional en los tres casos
    idx_exp = construir_indice(DOCUMENTO, tam, solape=int(tam * 0.15))
    for tipo, pregunta in PREGUNTAS_PRUEBA.items():
        respuesta, rec = rag(pregunta, idx_exp, top_k=3)
        filas.append({
            'Tamaño': tam,
            'Nº frags': len(idx_exp['fragmentos']),
            'Tipo': tipo,
            'Respuesta': respuesta,
        })
    print(f'  ✔ tamaño {tam} terminado')

resultados = pd.DataFrame(filas)

# Ajustes de pandas para ver el texto completo de las respuestas
pd.set_option('display.max_colwidth', None)
display(resultados.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))
print('✅ Experimento terminado. Compara las respuestas de cada tipo entre tamaños.')

### 📝 ¿Qué observas?

Antes de seguir, mira la tabla y piensa (lo usarás en la reflexión):

- En la pregunta **🧩 Síntesis** con fragmentos de 40 palabras, ¿aparecen el desayuno, el restaurante, el bar **y** el servicio de habitaciones? ¿Y con 300?
- En la pregunta **🔤 Literal**, ¿afecta el tamaño a que la contraseña sea correcta?
- ¿En algún caso el modelo inventa un dato que no está en la guía?

In [ ]:
# ============================================================
# CELDA 6.3 — Definir las funciones que usará la interfaz
# ============================================================
# Una función de Gradio recibe los valores de los componentes y devuelve los valores de salida.

def a_tabla(lista, frags, nombre_puntuacion):
    """Convierte [(idx, puntuación), ...] en un DataFrame legible."""
    return pd.DataFrame([{'Pos': pos, 'Frag': f'F{i}', nombre_puntuacion: round(s, 4),
                          'Texto': frags[i][:150] + '...'}
                         for pos, (i, s) in enumerate(lista, start=1)])

def responder_ui(pregunta, documento, tam, solape, top_k):
    if not pregunta.strip():
        return '⚠️ Escribe una pregunta.', None, None, None, None, ''
    tam, solape, top_k = int(tam), int(solape), int(top_k)   # Los sliders devuelven float
    # Limitamos el solape a la mitad del tamaño: evita el ValueError (solape >= tam) y que
    # un solape casi igual al tamaño genere cientos de fragmentos casi idénticos (muy lento).
    solape = min(solape, tam // 2)

    inicio = time.time()
    idx_ui = construir_indice(documento, tam, solape)   # Usa la caché si ya existe
    respuesta, rec = rag(pregunta, idx_ui, top_k=top_k)
    frags = idx_ui['fragmentos']

    info = (f'📦 {len(frags)} fragmentos de {tam} palabras (solape {solape}) · '
            f'⏱️ {time.time() - inicio:.1f} s')
    return (respuesta,
            a_tabla(rec['final'], frags, 'Reranker'),
            a_tabla(rec['semantica'], frags, 'Coseno'),
            a_tabla(rec['bm25'], frags, 'BM25'),
            a_tabla(rec['rrf'], frags, 'RRF'),
            info)

# Prueba directa de la función, sin interfaz
salida = responder_ui('¿Cuánto cuesta el parking?', DOCUMENTO, 120, 20, 3)
print(salida[0], '\n', salida[5])
print('✅ Función de la interfaz probada')

In [ ]:
# ============================================================
# CELDA 6.4 — Construir la interfaz con gr.Blocks
# ============================================================
import gradio as gr   # Fijado a 5.x en la celda 1.3 (en Gradio 6, css y theme pasan a launch())

CSS = """
#titulo {text-align: center}
#respuesta {border: 2px solid #84cc16; border-radius: 10px; padding: 12px; min-height: 80px}
"""

# gr.Blocks permite colocar componentes libremente con filas (Row), columnas (Column) y pestañas (Tab).
with gr.Blocks(title='RAG Híbrido', css=CSS, theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🔎 RAG Híbrido con Reranking\n'
                'Semántica (bge-m3) + BM25 → RRF → bge-reranker-v2-m3 → llama3.2',
                elem_id='titulo')

    with gr.Row():
        # ── Columna izquierda: entradas ─────────────────────────────
        with gr.Column(scale=1):
            pregunta = gr.Textbox(label='Pregunta', lines=2,
                                  placeholder='¿A qué hora abre la piscina?')
            # gr.Examples rellena la caja de texto al hacer clic en un ejemplo
            gr.Examples(examples=list(PREGUNTAS_PRUEBA.values()), inputs=pregunta)
            tam = gr.Slider(20, 400, value=120, step=10, label='Tamaño de fragmento (palabras)')
            solape = gr.Slider(0, 100, value=20, step=5, label='Solape (palabras)')
            top_k = gr.Slider(1, 6, value=3, step=1, label='Fragmentos que recibe el LLM (top_k)')
            boton = gr.Button('Preguntar', variant='primary')
            with gr.Accordion('📄 Documento (puedes pegar otro texto)', open=False):
                documento = gr.Textbox(value=DOCUMENTO, lines=12, show_label=False)

        # ── Columna derecha: salidas ────────────────────────────────
        with gr.Column(scale=2):
            respuesta = gr.Markdown(elem_id='respuesta')
            info = gr.Markdown()
            with gr.Tab('🥇 Fragmentos finales (reranker)'):
                t_final = gr.Dataframe(wrap=True)
            with gr.Tab('💭 Semántica'):
                t_sem = gr.Dataframe(wrap=True)
            with gr.Tab('🔤 BM25'):
                t_bm25 = gr.Dataframe(wrap=True)
            with gr.Tab('🔀 Fusión RRF'):
                t_rrf = gr.Dataframe(wrap=True)

    entradas = [pregunta, documento, tam, solape, top_k]
    salidas = [respuesta, t_final, t_sem, t_bm25, t_rrf, info]
    boton.click(responder_ui, inputs=entradas, outputs=salidas)      # Clic en el botón
    pregunta.submit(responder_ui, inputs=entradas, outputs=salidas)  # Enter en la caja de texto

print('✅ Interfaz construida (aún no lanzada)')

In [ ]:
# ============================================================
# CELDA 6.5 — Lanzar la interfaz dentro de Colab
# ============================================================
# Prueba los tres ejemplos moviendo el slider de tamaño (40, 120, 300).
# Mira las pestañas para ver en qué etapa entra o sale cada fragmento.

demo.launch(quiet=True)

print('✅ Interfaz visible arriba 👆')

## 🤔 PARTE 7 — Reflexión

Responde en esta misma celda (doble clic para editarla). Entre 2 y 5 líneas por pregunta.

1. **Conceptual:** ¿Qué diferencia hay entre la búsqueda por significado (coseno sobre embeddings) y la búsqueda por palabras clave (BM25)? Da un ejemplo de pregunta en la que cada una gane a la otra.

   *Tu respuesta:*

2. **Código:** En `fusion_rrf` se ignora la puntuación original de cada búsqueda y solo se usa la posición. ¿Por qué? ¿Qué pasaría si sumáramos directamente el coseno y la puntuación BM25?

   *Tu respuesta:*

3. **Experimento:** Según tu tabla de la Parte 6, ¿qué tamaño de fragmento funcionó mejor para cada tipo de pregunta? ¿Existe un tamaño "perfecto" para todo?

   *Tu respuesta:*

4. **Implicaciones:** Este sistema podría responder a clientes reales de un hotel. ¿Qué riesgos hay si el modelo inventa un precio o una norma? ¿Qué partes del pipeline ayudan a reducir ese riesgo?

   *Tu respuesta:*

5. **Comparativa:** ¿Qué ventajas y desventajas tiene añadir el reranker frente a pasar directamente los resultados de RRF al LLM? Piensa en calidad, velocidad y coste.

   *Tu respuesta:*

## 🏆 PARTE 8 — Retos opcionales

Cada reto tiene una celda vacía debajo. No están resueltos: solo tienes una pista para empezar.

- **Reto A — ¿Cuánto aporta cada pieza?** Compara las respuestas usando solo búsqueda semántica, solo BM25 y el pipeline completo con las tres preguntas de prueba.
  *Pista:* copia `recuperar` y haz que `fusion` sea solo `sem` (o solo `lex`), o que `final` sea `fusion[:top_k]` sin pasar por el reranker.

- **Reto B — Fragmentos que respetan secciones:** en vez de cortar cada N palabras, corta por secciones del documento (cada una empieza con un título en MAYÚSCULAS seguido de punto).
  *Pista:* `DOCUMENTO.split('\n\n')` separa por párrafos; prueba la pregunta de síntesis con esos fragmentos.

- **Reto C — Tu propio documento:** pega en la interfaz el reglamento de tu empresa, un artículo de Wikipedia o los apuntes de una asignatura y diseña tus propias preguntas literal, de paráfrasis y de síntesis.
  *Pista:* recuerda adaptar el texto de `SISTEMA` (ahora dice "asistente del Hotel Mirador del Lago") y vuelve a ejecutar la celda 5.1.

In [ ]:
# ============================================================
# CELDA 8.1 — Reto A: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

In [ ]:
# ============================================================
# CELDA 8.2 — Reto B: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

In [ ]:
# ============================================================
# CELDA 8.3 — Reto C: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

## 🌐 PARTE 9 — Publicación con Ngrok

Hasta ahora la app solo vive dentro de Colab. Con **Ngrok** creamos un **túnel** hacia Internet
para que tus compañeros la abran desde su navegador.

```
  Compañero (navegador)
        │  https://xxxx.ngrok-free.app
        ▼
  Servidor de Ngrok  ───túnel───►  Colab (localhost:7860)
                                          │
                                          ▼
                                   Tu app Gradio  ──►  Ollama + reranker
```

**Antes de empezar:** crea una cuenta gratuita en [ngrok.com](https://ngrok.com), copia tu *Authtoken*
y guárdalo en Colab: icono 🔑 **Secrets** → *Add new secret* → nombre `NGROK_TOKEN` → pega el token → activa *Notebook access*.

In [ ]:
# ============================================================
# CELDA 9.1 — Instalar pyngrok
# ============================================================
# Ya lo instalamos en la celda 1.3; repetirlo es instantáneo y garantiza que esté
# disponible aunque hayas reiniciado la sesión.

!pip install -q pyngrok

print('✅ pyngrok instalado')

In [ ]:
# ============================================================
# CELDA 9.2 — Autenticar con el token de Ngrok
# ============================================================
# El token es como una contraseña: NUNCA lo escribas directamente en el notebook
# si vas a compartirlo, porque cualquiera podría usar tu cuenta.
# Por eso lo leemos desde Colab Secrets.

from pyngrok import ngrok            # Librería para crear el túnel

try:
    from google.colab import userdata                 # Acceso a los Secrets de Colab
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')         # Lee el secreto llamado NGROK_TOKEN
except Exception as e:
    # Si Secrets no está disponible o el secreto no existe, usamos la variable manual.
    print(f'⚠️ No se pudo leer el Secret: {e}')
    print('⚠️ SEGURIDAD: si escribes el token aquí, bórralo antes de compartir el notebook.')
    NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUI'                # Solo como último recurso

# Validamos que no quedó el texto de ejemplo
if NGROK_TOKEN == 'PEGA_TU_TOKEN_AQUI':
    raise ValueError('❌ Falta el token. Guárdalo en Secrets con el nombre NGROK_TOKEN y reintenta.')

ngrok.set_auth_token(NGROK_TOKEN)   # Registra el token para esta sesión

print('✅ Token de Ngrok configurado')

In [ ]:
# ============================================================
# CELDA 9.3 — Abrir el túnel y relanzar la app
# ============================================================
# Orden importante: cerrar lo anterior → abrir túnel → lanzar la app.

demo.close()    # Cierra la app lanzada en la Parte 6 para liberar el puerto
ngrok.kill()    # Cierra túneles anteriores (el plan gratuito solo permite uno a la vez)

# connect(7860) crea un túnel público hacia el puerto local 7860
tunel = ngrok.connect(7860)
print(f'🌍 URL pública: {tunel.public_url}')
print('   Compártela con tus compañeros y ábrela en tu celular para comprobarla.')

# server_name='0.0.0.0' = acepta conexiones externas (las que llegan por el túnel)
# server_port=7860       = el mismo puerto que expusimos con Ngrok
# share=False            = no usamos el enlace temporal de Gradio; Ngrok es más estable
demo.launch(server_name='0.0.0.0', server_port=7860, share=False, quiet=True)

print('✅ App publicada con Ngrok')

In [ ]:
# ============================================================
# CELDA 9.4 — Verificar los túneles activos
# ============================================================
# get_tunnels() lista los túneles abiertos en esta sesión.

tuneles = ngrok.get_tunnels()

if tuneles:
    print(f'✅ {len(tuneles)} túnel(es) activo(s):')
    for t in tuneles:
        print(f'   {t.public_url}  →  {t.config["addr"]}')
else:
    print('❌ No hay túneles activos. Ejecuta de nuevo la celda 9.3.')

In [ ]:
# ============================================================
# CELDA 9.5 — (Opcional) Dominio estático
# ============================================================
# Con el plan gratuito puedes reclamar UN dominio fijo en dashboard.ngrok.com → Domains.
# Así la URL no cambia entre sesiones. Si no tienes uno, sáltate esta celda.

DOMINIO_ESTATICO = ''   # Ej: 'mi-rag-hibrido.ngrok-free.app'

if DOMINIO_ESTATICO:
    ngrok.kill()                                                    # Cierra el túnel aleatorio
    tunel = ngrok.connect(7860, domain=DOMINIO_ESTATICO)            # Abre el túnel con tu dominio
    print(f'✅ URL fija: {tunel.public_url}')
else:
    print('ℹ️ Sin dominio estático configurado: se mantiene la URL aleatoria.')

In [ ]:
# ============================================================
# CELDA LIMPIEZA — Liberar todos los recursos al terminar
# ============================================================
# Orden: túnel → interfaz → procesos → memoria GPU.

# 1. Cerrar el túnel Ngrok
ngrok.kill()
print('✅ Túnel Ngrok cerrado')

# 2. Cerrar la interfaz Gradio
demo.close()
print('✅ Interfaz Gradio cerrada')

# 3. Detener el servidor Ollama (libera también la memoria de bge-m3 y llama3.2)
proceso_ollama.terminate()
print('✅ Servidor Ollama detenido')

# 4. Liberar la memoria GPU del reranker
del reranker                 # Borra la referencia al modelo
_cache.clear()               # Vacía los índices guardados
torch.cuda.empty_cache()     # Devuelve a la GPU la memoria que PyTorch tenía reservada
print('✅ Memoria GPU liberada')

print('\n✅ Todos los recursos liberados. Puedes cerrar el notebook.')

## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|----------|---------------|-------------------|-------------------|
| **Preparación del entorno** | Parte 1 completa sin errores: Ollama, `bge-m3`, `llama3.2` y reranker cargados | Funciona tras algún reintento o reinicio | No logra descargar los modelos |
| **Fragmentación y búsquedas** | Explica el solape y muestra una pregunta donde gana BM25 y otra donde gana la semántica | Ejecuta las búsquedas pero no compara sus resultados | Las búsquedas no devuelven resultados |
| **Fusión y reranking** | Interpreta cómo cambia el orden entre semántica, BM25, RRF y reranker | Ejecuta el pipeline sin analizar las etapas | El pipeline falla |
| **Experimento de tamaños** | Tabla completa con conclusiones por tipo de pregunta y tamaño | Tabla completa sin conclusiones claras | Experimento incompleto |
| **Publicación con Ngrok** | URL pública funcionando y abierta desde otro dispositivo | Ngrok conecta pero sin verificar desde otro dispositivo | No intenta la publicación |
| **Reflexión y retos** | 5 respuestas razonadas e intenta al menos un reto | Respuestas incompletas o genéricas | Sin respuestas |